load document

In [1]:
! pip install -qU langchain-community pypdf

In [2]:
from google.colab import files

uploaded = files.upload()

Saving NIPS-2017-attention-is-all-you-need-Paper.pdf to NIPS-2017-attention-is-all-you-need-Paper (2).pdf


In [3]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/NIPS-2017-attention-is-all-you-need-Paper.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()
print(doc[5].metadata)

/tmp/ipykernel_80669/2854736560.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


{'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-French t

In [4]:
!pip install -qU langchain-text-splitters

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
)
all_splitts = text_splitter.split_documents(doc)
print(all_splitts[5].page_content)

Attention mechanisms have become an integral part of compelling sequence modeling and transduc-
tion models in various tasks, allowing modeling of dependencies without regard to their distance in
the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for signiﬁcantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Background
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[20], ByteNet [15] and ConvS2S [8], all of which use convolutional neural networks as basic building


In [6]:
!pip install -qU langchain langchain-huggingface sentence-transformers

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",

)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
!pip install -U langchain-chroma chromadb opentelemetry-api opentelemetry-sdk

In [9]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="research_collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"
)

document_ids = vector_store.add_documents(
    documents=all_splitts
)

print(document_ids)
sample = vector_store.get(limit = 1, include = ["embeddings","documents"])
print(sample)

['5f88f122-eb62-453e-8cd2-76182a3c2915', 'd04be93e-ef4f-436c-88ad-fe05ebf65f91', '76f0b138-a62c-4ecd-9bce-84517095a0a4', '1c5d7052-12c8-4f29-9f31-64dd1d990b3c', '1d457e22-e6a5-4edb-b56f-ae4a5c9b9fd5', '5cef20b9-86a8-468f-89fd-139f934bf3d6', 'ef88524c-0865-4352-9b92-c9112178a41a', 'bbc0351b-ea31-4846-b1d0-e26dbe5bb2a6', 'b0550600-5481-451f-9151-0c6307479b58', '90de0b88-6dc4-4cf7-8528-863892a158c1', '7063dbc8-d70d-4326-a4e7-4d71a4c8e902', 'b48dcb38-baf7-47d1-beca-3dd0d26e797d', '4dea206f-7896-4bf3-84d3-da9deeac03c2', 'e2563993-6367-4e77-b070-0675b882efb6', '860bcdf6-8ccb-4be8-8408-54f49639687a', '155722e1-a7fd-4e4d-a552-cc405530f1f4', '0006d9cb-b442-46a8-a75b-e9ecffb0214e', '32dcc13c-3e76-4bbc-b5ed-35833c5adfe5', '3c61c731-310e-4df2-aad0-0fac0d339525', '8fd825f7-df5f-4901-b052-134e64026c05', '74e586fa-916d-4af3-a204-ea46a43192e2', 'f6a53b31-658a-4422-8dac-817eb5f4fd47', '0b27b582-7116-403f-8cdb-c7212576eef4', '78316522-5c9f-48ec-8ba8-8a5d81eebd36', '0563f2d0-fcd9-4949-be0a-dd6d713feb86',

In [10]:
def retrieve_context(query: str, k: int = 2):
  retrieved_docs = vector_store.similarity_search(query, k=k)

  docs_content = ""
  for doc in retrieved_docs:
    docs_content += f"Source: {doc.metadata}\n"
    docs_content += f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs

In [11]:
!pip install -U langchain-google-genai

In [20]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
    "google_genai:gemini-3.6-flash",
    api_key=api_key,
)


In [17]:
def docu_chat(user_query):
    context, source_docs = retrieve_context(user_query, k=2)

    system_message = f"""You are a helpful chatbot.
    Use only the following pieces of context to answer the question.
    Don't makeup any new information: {context}"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_query}
    ]

    response = model.invoke(messages)

    return {
        "answer": response.content,
        "source_documents": source_docs,
        "context_used": context
    }


In [21]:
result = docu_chat("Explain what is the use of decoders in transformers?")
print(result)
print(result["answer"])

{'answer': [{'type': 'text', 'text': 'Based on the provided context, the decoder in a transformer model serves the following roles and functions:\n\n* **Encoder-Decoder Configuration:** It works alongside the encoder in a sequence transduction model (used for tasks like machine translation).\n* **Layer Structure:** It is composed of a stack of $N = 6$ identical layers.\n* **Attention over Encoder Output:** In addition to the sub-layers present in the encoder, the decoder inserts a third sub-layer that performs multi-head attention over the output of the encoder stack.\n* **Preventing Future Attending:** Its self-attention sub-layer is modified to prevent positions from attending to subsequent positions.\n* **Residual Connections and Normalization:** Like the encoder, it employs residual connections around each of its sub-layers, followed by layer normalization.', 'extras': {'signature': 'EtgNCtUNAWkUfRMEtBJRlTvgwj+9UF1FjJxc8HbouRNhL3Y7Y8FAwE+spelrUHB0ZKSX004iuQn59j22vphkUHw7OusGsdapBHS